# Exploration des Données Multi-Omiques (Pan-Cancer & BRCA)

Ce notebook utilise les nouveaux outils de chargement pour explorer le dataset Pan-Cancer fusionné avec les données cliniques BRCA.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import joblib
import importlib

# Robust path setup
project_root = Path(os.getcwd())
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Racine du projet : {project_root}")

# Import common components
from xai_clinical.data.pancan_loader import PANCANLoader, BRCALoader, PANCANBRCAFusion


## 1. Initialisation des Chargeurs

Nous configurons les chargeurs pour pointer vers les données brutes.

In [ ]:
pancan_dir = project_root / "data" / "raw" / "pancan"
brca_dir = project_root / "data" / "raw" / "brca_tcga"

# Initialisation (sans filtre pour charger tout le Pan-Cancer)
pancan_loader = PANCANLoader(str(pancan_dir), cancer_type=None)
brca_loader = BRCALoader(str(brca_dir))

## 2. Chargement des Données Cliniques (BRCA)

Ici, nous utilisons la correction qui sélectionne `PATIENT_ID` comme index.

In [ ]:
clinical_df = brca_loader.get_merged_clinical()
print(f"Données cliniques chargées : {clinical_df.shape}")
display(clinical_df.head(2))

# Extraction de la cible (Décès < 24 mois)
target = brca_loader.extract_survival_target(clinical_df, cutoff_months=24)
print("\nDistribution de la cible de survie :")
print(target.value_counts())

## 3. Chargement de l'Expression Génique (Pan-Cancer)

Attention : Le fichier fait 1.8 Go. Le chargement peut prendre du temps.

In [ ]:
expression_df = pancan_loader.load_gene_expression()
print(f"Données d'expression chargées : {expression_df.shape}")

## 4. Fusion et Alignement

On aligne les patients communs entre les gènes (Pan-Cancer) et la clinique (BRCA).

In [ ]:
fusion = PANCANBRCAFusion(pancan_loader, brca_loader)
fused_df = fusion.fuse_expression_clinical(expression_df, clinical_df, target)

print(f"Données fusionnées : {fused_df.shape}")
if 'TARGET' in fused_df.columns:
    print("\nDistribution finale dans le dataset fusionné :")
    print(fused_df['TARGET'].value_counts())

## 5. Visualisation Rapide

Comparaison de l'expression de quelques gènes clés (ex: ESR1 pour le cancer du sein) selon la survie.

In [ ]:
esr1_col = next((c for c in fused_df.columns if c.startswith('ESR1|')), 'ESR1')
    if esr1_col in fused_df.columns and 'TARGET' in fused_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='TARGET', y=esr1_col, data=fused_df)
    plt.title("Expression de ESR1 vs Survie (0=Vivant, 1=Décès < 24m)")
    plt.show()
else:
    print("ESR1 non trouvé dans les colonnes ou fusion échouée.")
